# Notebook 03: The Agent's Brain — Planning

Tools give an agent hands. Planning gives it a brain — the ability to decompose complex tasks into steps, decide what to do when, and recover from failures.

**Expected time:** ~60 minutes  
**Prerequisites:** Notebooks 00-02

```mermaid
graph TD
    Goal([Task Goal]) --> Plan[Plan: Decompose into Steps 1..N]
    Plan --> Check{More steps left?}
    Check -->|Yes| Exec[Execute step N using Tool]
    Exec --> Validate{Is step result valid?}
    Validate -->|Yes| Next[Increment N] --> Check
    Validate -->|No| Fix[Revise Plan / Retry step] --> Exec
    Check -->|No| Done([Return final result])
```


## What You'll Learn

1. Chain-of-Thought: how LLMs decompose problems naturally
2. Task decomposition patterns: linear, parallel, conditional
3. Plan-then-execute: generate a plan, then follow it with tools
4. Multi-tool routing: agent decides *which* tool to use
5. Common planning failures and how to catch them


## Setup


In [1]:
!pip install openai==1.68.2

from agent_helpers import ReactAgent, make_tool, mock_llm, safe_calc
import os
import getpass
import json
from pathlib import Path

# --- Optional .env file loading (no extra pip installs needed) ---
env_path = Path.cwd() / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    print("Loaded .env file.")
else:
    print("No .env file found — you will be prompted for API keys.")
    print("  Tip: cp .env.example .env  and fill in your keys to skip prompts.")

# --- Pick your provider ---
PROVIDER = "openrouter"  # Choose: openrouter, groq, openai
# --------------------------

MOCK = {
    "openrouter": False,
    "groq":       True,
    "openai":     True,
}

KEY_ENV_VAR = {
    "openrouter": "OPENROUTER_API_KEY",
    "groq":       "GROQ_API_KEY",
    "openai":     "OPENAI_API_KEY",
}

CONFIG = {
    "openrouter": {
        "name": "OpenRouter",
        "model": "deepseek/deepseek-v4-flash",
        "base_url": "https://openrouter.ai/api/v1",
        "key_url": "https://openrouter.ai/keys",
    },
    "groq": {
        "name": "Groq",
        "model": "llama-3.3-70b-versatile",
        "base_url": "https://api.groq.com/openai/v1",
        "key_url": "https://console.groq.com/keys",
    },
    "openai": {
        "name": "OpenAI",
        "model": "gpt-4o-mini",
        "base_url": None,
        "key_url": "https://platform.openai.com/api-keys",
    },
}

cfg = CONFIG[PROVIDER]
use_mock = MOCK[PROVIDER]

if use_mock:
    print(f"[{cfg['name']}] Mock mode enabled.")
    client = None
    model = mock_llm()
else:
    from openai import OpenAI
    api_key = os.environ.get(KEY_ENV_VAR[PROVIDER]) or getpass.getpass(f"Enter your {cfg['name']} API key: ")
    kwargs = {"api_key": api_key}
    if cfg["base_url"]:
        kwargs["base_url"] = cfg["base_url"]
    client = OpenAI(**kwargs)

    def make_model(client, model=None):
        if model is None:
            model = cfg["model"]
        def fn(messages):
            try:
                r = client.chat.completions.create(model=model, messages=messages, max_tokens=500)
                if not r.choices:
                    return "Error: empty response from API"
                content = r.choices[0].message.content
                return content if content is not None else ""
            except Exception as e:
                return f"API Error: {type(e).__name__}: {e}"
        return fn

    model = make_model(client)
    print(f"[{cfg['name']}] API key configured. Model: {cfg['model']}")


Loaded .env file.
[OpenRouter] API key configured. Model: deepseek/deepseek-v4-flash


## Chain of Thought (CoT)

Chain of Thought is the simplest form of planning: tell the LLM to "think step by step" and it decomposes a complex question into intermediate reasoning steps.

No tools yet — just reasoning. This builds the habit of structured thinking before we add tool execution.

In [2]:
query = "A student has 3 lectures, 1 lab, and wants to study 4 hours. Each lecture is 90 min, lab is 2 hours. How much free time do they have in a 12-hour day?"

if not use_mock and client is not None:
    with_cot = client.chat.completions.create(
        model=cfg["model"],
        messages=[
            {"role": "system", "content": "Think step by step. Show your reasoning."},
            {"role": "user", "content": query},
        ],
    )
    print("WITH Chain of Thought:")
    print(with_cot.choices[0].message.content)
else:
    print("Mock — CoT response:")
    print("  Step 1: Convert everything to hours.")
    print("  Step 2: 3 lectures × 1.5h = 4.5h")
    print("  Step 3: 1 lab = 2h")
    print("  Step 4: Study = 4h")
    print("  Step 5: Total busy = 4.5 + 2 + 4 = 10.5h")
    print("  Step 6: Free = 12 - 10.5 = 1.5h")


WITH Chain of Thought:
Let's go step by step.

**Step 1: Convert everything to the same unit (hours).**  
- Each lecture: 90 minutes = 1.5 hours  
- There are 3 lectures → \( 3 \times 1.5 = 4.5 \) hours  
- Lab: 2 hours  
- Study time: 4 hours  

**Step 2: Add up total time used for lectures, lab, and study.**  
\[
4.5 \text{ (lectures)} + 2 \text{ (lab)} + 4 \text{ (study)} = 10.5 \text{ hours}
\]

**Step 3: Subtract from total day length.**  
Total day: 12 hours  
Free time = \( 12 - 10.5 = 1.5 \) hours  

\[
\boxed{1.5}
\]  

They have 1.5 hours of free time.


In [3]:
if not use_mock and client is not None:
    without_cot = client.chat.completions.create(
        model=cfg["model"],
        messages=[
            {"role": "user", "content": query + "\nAnswer in one sentence."},
        ],
    )
    print("WITHOUT Chain of Thought:")
    print(without_cot.choices[0].message.content)
    print("\nCoT produces more reliable, debuggable answers.")
else:
    print("Mock — without CoT: They have 1.5 hours free.")
    print("Shorter, but you can't verify the reasoning.")


WITHOUT Chain of Thought:
The student has 3 lectures of 90 minutes each, totaling 4.5 hours, one lab of 2 hours, and wants to study for 4 hours. The total time spent is 4.5 + 2 + 4 = 10.5 hours. In a 12-hour day, the free time is 12 - 10.5 = 1.5 hours.

\boxed{1.5} \text{ hours}

CoT produces more reliable, debuggable answers.


## Task Decomposition Patterns

Planning problems come in three shapes. Let's see each with a Mermaid diagram and a concrete example.


### Pattern 1: Linear Decomposition

Steps run in sequence. Each depends on the previous.

```mermaid
graph LR
    A[Research] --> B[Draft] --> C[Review] --> D[Publish]
```

*Example: "Write a summary of agentic AI."*

In [4]:
def linear_planner(task, model_fn):
    prompt = (
        f"Break this task into exactly 3 sequential steps.\n"
        f"Format each step as: STEP: <short description>\n"
        f"Task: {task}"
    )
    return model_fn([{"role": "user", "content": prompt}])

plan = linear_planner("Write a summary of agentic AI for beginners", model)
print("Linear plan:")
print(plan)


Linear plan:
STEP: Research and understand the core concepts of agentic AI, including autonomy, goals, and decision-making.
STEP: Outline key points tailored for beginners, such as examples and comparisons to traditional AI.
STEP: Write a clear, concise summary that explains agentic AI in simple terms without jargon.


### Pattern 2: Parallel Decomposition

Independent tasks run in parallel, then merge results.

```mermaid
graph TD
    A[Research angle] --> D[Merge & Write]
    B[Research angle 2] --> D
    C[Research angle 3] --> D
```

*Example: "Research pros, cons, and trends of agentic AI, then combine."*

In [5]:
def parallel_planner(task, model_fn):
    prompt = (
        f"Break this task into 3 independent sub-tasks that can run in parallel, "
        f"followed by a merge step.\n"
        f"Format: PARALLEL: <step>, PARALLEL: <step>, PARALLEL: <step> → MERGE: <step>\n"
        f"Task: {task}"
    )
    return model_fn([{"role": "user", "content": prompt}])

plan = parallel_planner("Research pros, cons, and future trends of agentic AI", model)
print("Parallel plan:")
print(plan)


Parallel plan:
PARALLEL: Research the pros (advantages, benefits, use cases) of agentic AI, PARALLEL: Research the cons (risks, limitations, ethical concerns) of agentic AI, PARALLEL: Research the future trends (emerging technologies, market forecasts, research directions) of agentic AI → MERGE: Synthesize the findings from all three sub-tasks into a comprehensive, structured summary covering pros, cons, and future trends.


### Pattern 3: Conditional Decomposition

The path depends on a condition. Different inputs take different routes.

```mermaid
graph TD
    Q[Query type?] -->|Math| Calc[Use calculator]
    Q -->|Weather| Weather[Check weather API]
    Q -->|General| Search[Search web]
```

*Example: "Route user query to the right tool based on type."*

In [6]:
def conditional_planner(task, model_fn):
    prompt = (
        f"Classify this request into one category: math, weather, search, or other.\n"
        f"Then suggest exactly ONE tool to handle it.\n"
        f"Format: CATEGORY: <type> → TOOL: <tool>\n"
        f"Request: {task}"
    )
    return model_fn([{"role": "user", "content": prompt}])

for task in ["What is 2+2?", "Weather in Copenhagen", "Latest AI news"]:
    print(f">>> {task}")
    print(conditional_planner(task, model))
    print()


>>> What is 2+2?
CATEGORY: math → TOOL: calculator

>>> Weather in Copenhagen
CATEGORY: weather → TOOL: OpenWeatherMap API

>>> Latest AI news
CATEGORY: search → TOOL: Google



## Plan-then-Execute

Now we connect planning to execution. The agent:
1. Generates a plan
2. Executes each step using tools
3. Validates each step result
4. Moves to next step or retries

This is **Complexity Ladder Level 3**: the agent makes decisions about what to do next.

### Tools for Planning


In [7]:
def search_web(query):
    db = {
        "agentic ai trends 2026": "Agentic AI is shifting from chat to autonomous task completion. Major frameworks: LangGraph, CrewAI, AutoGen.",
        "python machine learning": "Scikit-learn, PyTorch, and TensorFlow dominate. New: small language models running locally.",
        "pbl project ideas": "Common themes: research assistant agents, personal scheduling agents, data analysis pipelines.",
    }
    return db.get(query.lower(), f"No stored results for: {query}")

def write_note(note):
    return f"Saved note: {note[:50]}..."

tools = [
    make_tool("search", "Search web for information", search_web),
    make_tool("calculator", "Evaluate math expressions", safe_calc),
    make_tool("write_note", "Save a note", write_note),
]
print("Planning tools ready.")


Planning tools ready.


### Plan → Execute Loop

This function generates a plan and executes each step. It validates each step's output before proceeding.

In [8]:
def plan_and_execute(task, model_fn, tools, max_steps=8):
    """Generate plan, execute each step, validate, proceed."""
    plan_prompt = (
        f"Create a step-by-step plan for this task.\n"
        f"Format each step: STEP <N>: <action> | tool: <tool_name> | input: <args>\n"
        f"Available tools: calculator, search, write_note\n"
        f"Task: {task}"
    )
    plan_text = model_fn([{"role": "user", "content": plan_prompt}])
    print("GENERATED PLAN:")
    print(plan_text)
    print("\n--- EXECUTING ---\n")

    agent = ReactAgent(model=model_fn, tools=tools, max_steps=max_steps)
    result = agent.run(task)
    print("FINAL RESULT:")
    print(result)
    return result


In [9]:
task = "Research latest agentic AI trends and write a short summary note"

if not use_mock and client is not None:
    plan_and_execute(task, model, tools)
else:
    print("Mock plan-execute flow:")
    print("  STEP 1: Search for agentic AI trends | tool: search | input: agentic AI trends 2026")
    print("  STEP 2: Process search results")
    print("  STEP 3: Write summary note | tool: write_note | input: ...")
    print("\nPlan validation: each step verified before next.")


GENERATED PLAN:
STEP 1: Search for the latest trends in agentic AI. | tool: search | input: "latest agentic AI trends 2025"  

STEP 2: Based on search results, identify key trends, frameworks, and notable developments. | tool: search | input: "agentic AI frameworks 2025"  

STEP 3: Refine search to include specific industry applications or notable research. | tool: search | input: "agentic AI applications autonomy multi-agent 2025"  

STEP 4: Compile the key findings into a short summary. | tool: write_note | input: "Agentic AI Trends Summary (2025): [List key trends, frameworks, and applications found in searches]"

--- EXECUTING ---

FINAL RESULT:
Here is a short summary note on the latest trends in Agentic AI, based on research through early 2025.

---

**Subject: Summary of Latest Agentic AI Trends (Q1 2025)**

**To:** [Team/Stakeholders]
**From:** [Your Name]
**Date:** [Current Date]

**Key Takeaway:** Agentic AI is moving rapidly from proof-of-concept prototypes to structured, en

### Step Validation

A key improvement: check each step's output before moving on. If a step fails, the agent retries or adjusts.

In [10]:
def validated_plan(task, model_fn, tools, max_retries=2):
    """Plan with per-step validation and retry."""
    prompt = f"Create 3-4 numbered steps for: {task}"
    plan = model_fn([{"role": "user", "content": prompt}])
    print("Plan:", plan, "\n")

    if not plan or not isinstance(plan, str) or plan.startswith("API Error:"):
        print(f"  ✗ Plan generation failed: {plan}")
        return ""

    import re
    steps = []
    current = []
    for line in plan.split("\n"):
        s = line.strip()
        if re.match(r'^\d+[.\)] ', s):
            if current:
                steps.append(" ".join(current))
            current = [s]
        elif current and s:
            current.append(s)
    if current:
        steps.append(" ".join(current))
    if not steps:
        steps = [s for s in plan.split("\n") if s.strip()][:4]
    print(f"Executing {len(steps)} steps...\n")

    results = []
    history = []

    for i, step in enumerate(steps):
        for attempt in range(max_retries + 1):
            ctx = f"Your overall task: {task}"
            if history:
                ctx += "\nCompleted so far:\n" + "\n".join(history)
            result = model_fn([
                {"role": "system", "content": ctx},
                {"role": "user", "content": f"Execute step {i+1}: {step}"}
            ])
            validation = model_fn([
                {"role": "system", "content": "Reply ONLY with YES or NO."},
                {"role": "user", "content":
                 f"Did the result complete the step?\nStep: {step}\nResult: {result}"}
            ])
            if "YES" in validation.upper():
                results.append(result)
                history.append(f"Step {i+1}: completed")
                print(f"  ✓ Step {i+1} passed")
                break
            else:
                print(f"  ✗ Step {i+1} failed (attempt {attempt+1}), retrying...")
        else:
            print(f"  ✗ Step {i+1} failed after {max_retries} retries, skipping")

    return "\n".join(results)

if not use_mock and client is not None:
    validated_plan("Explain the benefits and risks of agentic AI in healthcare", model, tools)
else:
    print("Mock — validation flow:")
    print("  ✓ Step 1 passed")
    print("  ✓ Step 2 passed")
    print("  ✗ Step 3 failed (attempt 1), retrying...")
    print("  ✓ Step 3 passed (attempt 2)")
    print("\nValidation prevents garbage-in-garbage-out through the plan.")


Plan: Here are 3-4 steps explaining the benefits and risks of agentic AI in healthcare:

1.  **Benefit: Enhanced Diagnostic Accuracy and Personalized Treatment**  
    Agentic AI can autonomously analyze vast datasets (e.g., medical imaging, genomic data, patient history) to detect patterns missed by humans, leading to earlier and more precise diagnoses. It can then propose tailored treatment plans, optimizing drug dosages or therapy schedules for individual patients, improving outcomes while reducing trial-and-error.

2.  **Risk: Loss of Human Oversight and Accountability**  
    Autonomous decision-making—such as adjusting a patient’s medication or prioritizing emergency cases—creates a “black box” problem. If an AI makes a harmful recommendation, it becomes unclear who is liable: the developer, the hospital, or the clinician. This can erode trust and complicate legal and ethical responsibility in critical care.

3.  **Benefit: Operational Efficiency and Reduced Clinician Burnout**  

## Multi-Tool Routing

A more advanced pattern: the agent looks at the query, decides what *kind* of task it is, and routes to the appropriate tool automatically.

In [11]:
def router_agent(query, model_fn):
    """Classify query and call the right tool."""
    classification = model_fn([{"role": "user", "content": (
        f"Classify this query as one of: math, weather, file, search, other.\n"
        f"Reply with just the category.\n"
        f"Query: {query}"
    )}])

    category = classification.strip().lower()

    if category == "math":
        return "ROUTING → calculator"
    elif category == "weather":
        return "ROUTING → weather tool"
    elif category == "search":
        return "ROUTING → web search"
    else:
        return f"ROUTING → general LLM response"

for q in ["What is 144 / 12?", "Weather in Tokyo", "Tell me about PBL"]:
    print(f">>> {q}")
    print(f"    {router_agent(q, model)}")


>>> What is 144 / 12?
    ROUTING → calculator
>>> Weather in Tokyo
    ROUTING → weather tool
>>> Tell me about PBL
    ROUTING → web search


## What Breaks?


### Break 1: Circular Plan Dependencies

Step A needs Step B's output, but Step B needs Step A's output. The agent can never finish.

In [12]:
def circular_plan(task, model_fn):
    prompt = f"Create a plan for: {task}. Make step 2 depend on step 1, and step 1 depend on step 2."
    plan = model_fn([{"role": "user", "content": prompt}])
    print("Problematic plan:")
    print(plan)
    print("\nFix: Before execution, check for circular deps by building a dependency graph.")

circular_plan("Analyze survey data", model)


Problematic plan:


Fix: Before execution, check for circular deps by building a dependency graph.


### Break 2: Plan Too Fine-Grained

Too many tiny steps (20+) cause the agent to lose track of where it is. The context window fills with partial results.

In [13]:
def fine_grained_plan(task, model_fn):
    prompt = f"Create a very detailed plan with 15-20 micro-steps for: {task}"
    plan = model_fn([{"role": "user", "content": prompt}])
    steps = [s for s in plan.split("\n") if s.strip()]
    print(f"Generated {len(steps)} steps.")
    print("First 3:", steps[:3] if steps else "none")
    print("Last 3:", steps[-3:] if steps else "none")
    print("\nFix: Group micro-steps into 3-5 larger phases. "
          "Keep plans at the right abstraction level.")

fine_grained_plan("Research and summarize a topic", model)


Generated 19 steps.
First 3: ['Here is a detailed, actionable plan with 17 micro-steps for researching and summarizing any topic. Each step is designed to be specific and repeatable.', '---', '### Phase 1: Foundation & Focus (Steps 1–4)']
Last 3: ['   Check Currency, Relevance, Authority, Accuracy, and Purpose. Discard any that fail on authority or clear bias (e.g., opinion pieces pretending to be facts).', '7. **Select your final 5–7 core sources.**  ', '   Ensure a mix: at least one overview, one recent study (last 3 years), one contrasting viewpoint,']

Fix: Group micro-steps into 3-5 larger phases. Keep plans at the right abstraction level.


### Break 3: Plan Too Coarse

A one-step plan like "solve the problem" defeats the purpose of planning.

In [14]:
def coarse_plan(task, model_fn):
    prompt = f"Create a plan for: {task}. Make it exactly 1-2 very broad steps."
    plan = model_fn([{"role": "user", "content": prompt}])
    print("Overly broad plan:")
    print(plan)
    print("\nFix: Each step should be a single, executable action. If a step needs sub-steps, it is too broad.")

coarse_plan("Build an AI agent", model)


Overly broad plan:
Here is a plan to build an AI agent, broken into exactly two very broad steps.

**Step 1: Define the Agent’s Core "Brain" & World Model**
- Select or build the underlying AI model (e.g., a large language model, a reinforcement learning policy, or a hybrid system).
- Define the agent’s **perception** (how it receives inputs: text, images, sensor data, API calls), its **reasoning** (how it processes that data to form goals or plans), and its **memory** (short-term context, long-term knowledge base).
- Establish the rules and boundaries for its actions (its "world model" or constraints).

**Step 2: Build the Action Loop & Integration Layer**
- Create the execution framework that allows the agent to act autonomously: decide, act, observe feedback, then learn or adapt.
- Integrate tools and interfaces (e.g., code execution, database queries, web APIs, robotic actuators, or user interaction channels) so the agent can perform tasks in its environment.
- Implement a feedback

## Connection to Agentic AI

Planning is what separates reactive systems from truly agentic ones. A system that only responds to the current input (even with tools) is still reactive. A system that generates and follows a plan is proactively working toward a goal.

**Complexity Ladder mapping:**
- **Level 2:** CoT decomposition + simple tool (you did this with linear_planner)
- **Level 3:** Plan-then-execute with validation (you did this with validated_plan)
- **Level 4:** Multiple cooperating agents with plans (Notebook 05)


### PBL Reflection


In [15]:
print("Think about YOUR project:")
print("  1. What task does your agent need to decompose into sub-steps?")
print("  2. Which planning pattern fits: linear, parallel, or conditional?")
print("  3. Where might your plan break? Circular deps? Too many steps?")
print("  4. What's your validation strategy — how will you know a step succeeded?")
print("\nWrite in your project journal.")


Think about YOUR project:
  1. What task does your agent need to decompose into sub-steps?
  2. Which planning pattern fits: linear, parallel, or conditional?
  3. Where might your plan break? Circular deps? Too many steps?
  4. What's your validation strategy — how will you know a step succeeded?

Write in your project journal.


## Exercises


### Bronze — Run Planner on Your Domain

Change `my_task` to a problem from your own field. Run the linear planner and observe how it decomposes.

In [16]:
# --- Edit this ---
my_task = "Analyze patient cognitive load data to detect early burnout indicators"
# -----------------

plan = linear_planner(my_task, model)
print(f"Plan for:\n  {my_task}\n")
print(plan)


Plan for:
  Analyze patient cognitive load data to detect early burnout indicators

STEP: Collect and preprocess patient cognitive load data from sources such as wearable sensors, task performance logs, or self-report surveys.  
STEP: Extract and analyze key metrics (e.g., reaction time variability, error rates, physiological signals) to identify patterns associated with cognitive overload.  
STEP: Compare extracted patterns against established burnout thresholds or baseline norms to flag early indicators of burnout.


### Silver — Add Conditional Routing

Extend the router_agent to handle more categories. Add "database" and "image" as new routing options.

In [17]:
# --- Extend the router ---
def extended_router(query, model_fn):
    classification = model_fn([{"role": "user", "content": (
        f"Classify this query as: math, weather, search, database, image, or other.\n"
        f"Reply with just the category.\n"
        f"Query: {query}"
    )}])
    category = classification.strip().lower()
    router = {
        "math": "→ calculator tool",
        "weather": "→ weather API",
        "search": "→ web search",
        "database": "→ SQL query tool",
        "image": "→ image analysis pipeline",
    }
    return f"ROUTING {router.get(category, f'→ general LLM ({category})')}"
# --------------------------

for q in ["Find user by email", "Analyze this MRI scan", "What is 2+2?"]:
    print(f">>> {q}")
    print(f"    {extended_router(q, model)}")


>>> Find user by email
    ROUTING → SQL query tool
>>> Analyze this MRI scan
    ROUTING → image analysis pipeline
>>> What is 2+2?
    ROUTING → calculator tool


### Gold — Plan with Self-Check Between Steps

Build a planning agent that generates a plan, executes each step, and validates the result before proceeding. If a step fails, the agent re-plans the remaining steps.

This integrates planning (Notebook 03) with reflection (Notebook 04 pattern).

In [18]:
class AdaptivePlanner:
    """Plan, execute, check, and re-plan on failure."""
    def __init__(self, model_fn, tools):
        self.model = model_fn
        self.tools = tools

    def run(self, task):
        plan = self.model([{"role": "user", "content": (
            f"Create a plan for this task. List each step as a separate line.\n"
            f"Task: {task}"
        )}])
        print("=== PLAN ===")
        print(plan, "\n")

        steps = [s.strip() for s in plan.split("\n") if s.strip() and not s.startswith("#")]
        results = []

        for i, step in enumerate(steps, 1):
            print(f"--- Step {i}: {step[:60]}...")
            result = self.model([{
                "role": "user",
                "content": f"Execute this step: {step}\nPrevious results: {results}"
            }])

            check = self.model([{"role": "user", "content": (
                f"Did this successfully complete the step? Answer YES or NO.\n"
                f"Step: {step}\nResult: {result}"
            )}])

            if "YES" in check:
                print(f"  ✓ Step {i} complete")
                results.append(f"Step {i} ({step[:30]}...): {result[:100]}")
            else:
                print(f"  ✗ Step {i} failed. Re-planning from here...")
                new_plan = self.model([{"role": "user", "content": (
                    f"The step failed: {step}\n"
                    f"Result was: {result}\n"
                    f"Suggest an alternative approach for this step:"
                )}])
                print(f"  → Alternative: {new_plan[:80]}")
                results.append(f"Step {i} (alternative): {new_plan[:100]}")

        return results

if not use_mock and client is not None:
    planner = AdaptivePlanner(model, tools)
    planner.run("Research Python ML frameworks and write a summary note")
else:
    print("Gold exercise — AdaptivePlanner class defined.")
    print("Test it with real API mode.")


=== PLAN ===
- Define the scope: Determine which Python ML frameworks to cover (e.g., TensorFlow, PyTorch, scikit-learn, JAX, etc.) and the focus (e.g., features, ease of use, performance, community).
- Gather initial information: Search official documentation, recent articles, and community resources for each selected framework.
- Compare key aspects: Note strengths, weaknesses, primary use cases, learning curves, and ecosystem support for each framework.
- Outline the summary: Structure the note with an introduction, framework descriptions, a comparison table or bullet points, and a conclusion with recommendations.
- Draft the summary: Write concise paragraphs covering each framework, highlighting unique features and typical applications.
- Review and refine: Check for accuracy, clarity, and consistency; ensure the note meets the intended audience's needs (e.g., beginners or experienced developers).
- Finalize and save: Format the summary note (e.g., Markdown or plain text) and store

## Next Steps

You now understand how agents plan: decompose tasks, route between tools, validate progress, and adapt on failure.

**Next notebook:** `04_reflection.ipynb` — "The Agent's Critical Eye"

> "Planning tells the agent what to do. Reflection tells it if it did well — and how to improve next time."
